# 10_Scatterplot_VAF_threshold

This notebook makes per-sample scatterplots comparing Mutect2 and DeepVariant calls from the merged exact-variant CSVs.

For each sample, it plots:

1. Mutect2 VAF vs DeepVariant VAF
2. Mutect2 alternate depth vs DeepVariant alternate depth
3. Mutect2 total depth vs DeepVariant total depth

By default, the scatterplots use only variants found by **both** callers, because caller-only variants do not have both x and y values. The summary table still counts variants found by both callers, Mutect2 only, and DeepVariant only.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
base_dir = Path(r"/project/knathans_shared/donetski/Notebooks/InputFiles/09_merged_HGVSg_HGVSp_by_exact_variant")
output_dir = base_dir / "10_scatterplot_VAF_threshold_by_sample"
output_dir.mkdir(parents=True, exist_ok=True)

# Use None to process every sample, or an integer like 5 for testing.
sample_limit = None

# Only rows found by both callers are plotted.
# Caller-only variants are still counted in the summary CSV.
plot_both_callers_only = True

# Scatterplot thresholds. These are tested independently, not as combo filters yet.
vaf_thresholds = [0.01, 0.05, 0.10]
altdepth_thresholds = [1, 3, 5, 7, 10]

include_no_threshold = True
include_independent_thresholds = True
include_combo_thresholds = True

# Print plots inline in the notebook.
# False is cleaner when looping through many samples/thresholds.
show_plots = False

## Find input CSVs

Each sample has a merged exact-variant CSV inside its sample folder.

In [ ]:
sample_files = sorted(base_dir.glob("*/*_combined_HGVSg_HGVSp_unique_exact_variants.csv"))

if sample_limit is not None:
    sample_files = sample_files[:sample_limit]

print("Sample files found:", len(sample_files))
for path in sample_files[:5]:
    print(path)


## Plot settings

These are the comparison columns expected in each merged CSV.


In [ ]:
plots = [
    {
        "plot_name": "vaf",
        "x_col": "Mutect2_VAF",
        "y_col": "DeepVariant_VAF",
        "x_label": "Mutect2 VAF",
        "y_label": "DeepVariant VAF",
        "axis_min": 0,
        "axis_max": 1,
    },
    {
        "plot_name": "alt_depth",
        "x_col": "Mutect2_alt_depth",
        "y_col": "DeepVariant_alt_depth",
        "x_label": "Mutect2 alternate depth",
        "y_label": "DeepVariant alternate depth",
        "axis_min": 0,
        "axis_max": None,
    },
    {
        "plot_name": "total_depth",
        "x_col": "Mutect2_total_depth",
        "y_col": "DeepVariant_total_depth",
        "x_label": "Mutect2 total depth",
        "y_label": "DeepVariant total depth",
        "axis_min": 0,
        "axis_max": None,
    },
]

required_columns = [
    "Mutect2_found",
    "DeepVariant_found",
    "Mutect2_VAF",
    "DeepVariant_VAF",
    "Mutect2_alt_depth",
    "DeepVariant_alt_depth",
    "Mutect2_total_depth",
    "DeepVariant_total_depth",
]


# Threshold-Setting Builder

In [ ]:
threshold_settings = []

if include_no_threshold:
    threshold_settings.append({
        "threshold_type": "none",
        "threshold_label": "no_threshold",
        "vaf_threshold": None,
        "altdepth_threshold": None,
    })

if include_independent_thresholds:
    for vaf_threshold in vaf_thresholds:
        threshold_settings.append({
            "threshold_type": "vaf_only",
            "threshold_label": f"vaf_ge_{str(vaf_threshold).replace('.', 'p')}",
            "vaf_threshold": vaf_threshold,
            "altdepth_threshold": None,
        })

    for altdepth_threshold in altdepth_thresholds:
        threshold_settings.append({
            "threshold_type": "altdepth_only",
            "threshold_label": f"altdepth_ge_{altdepth_threshold}",
            "vaf_threshold": None,
            "altdepth_threshold": altdepth_threshold,
        })

if include_combo_thresholds:
    for altdepth_threshold in altdepth_thresholds:
        for vaf_threshold in vaf_thresholds:
            threshold_settings.append({
                "threshold_type": "combo",
                "threshold_label": f"altdepth_ge_{altdepth_threshold}_vaf_ge_{str(vaf_threshold).replace('.', 'p')}",
                "vaf_threshold": vaf_threshold,
                "altdepth_threshold": altdepth_threshold,
            })

## Plotting function

The diagonal line shows where Mutect2 and DeepVariant would have exactly matching values. For the VAF plot, the optional threshold lines show the selected VAF cutoff on both axes.


In [ ]:
def is_found(series):
    return series.astype(str).str.upper().eq("Y")


def format_threshold(value):
    return str(value).replace(".", "p")


def make_threshold_configs():
    configs = [{
        "threshold_type": "none",
        "threshold_value": None,
        "threshold_label": "no_threshold",
        "plot_names": ["vaf", "alt_depth", "total_depth"],
    }]

    for threshold in vaf_thresholds:
        configs.append({
            "threshold_type": "vaf",
            "threshold_value": threshold,
            "threshold_label": f"vaf_ge_{format_threshold(threshold)}",
            "plot_names": ["vaf"],
        })

    for threshold in altdepth_thresholds:
        configs.append({
            "threshold_type": "alt_depth",
            "threshold_value": threshold,
            "threshold_label": f"altdepth_ge_{threshold}",
            "plot_names": ["alt_depth"],
        })

    return configs

def apply_thresholds(df, threshold_info):
    out_df = df.copy()

    vaf_threshold = threshold_info["vaf_threshold"]
    altdepth_threshold = threshold_info["altdepth_threshold"]

    if vaf_threshold is not None:
        out_df = out_df[
            (out_df["Mutect2_VAF"] >= vaf_threshold) &
            (out_df["DeepVariant_VAF"] >= vaf_threshold)
        ].copy()

    if altdepth_threshold is not None:
        out_df = out_df[
            (out_df["Mutect2_alt_depth"] >= altdepth_threshold) &
            (out_df["DeepVariant_alt_depth"] >= altdepth_threshold)
        ].copy()

    return out_df

def plot_sample(sample_file):
    sample_id = sample_file.parent.name
    df = pd.read_csv(sample_file, low_memory=False)

    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        print(f"Skipping {sample_id}; missing columns: {missing}")
        return []

    for column in required_columns[2:]:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    mutect2_found = is_found(df["Mutect2_found"])
    deepvariant_found = is_found(df["DeepVariant_found"])
    both_found = mutect2_found & deepvariant_found

    if plot_both_callers_only:
        base_df = df.loc[both_found].copy()
    else:
        base_df = df.copy()

    sample_summaries = []

    for threshold_config in make_threshold_configs():
        threshold_type = threshold_config["threshold_type"]
        threshold_value = threshold_config["threshold_value"]
        threshold_label = threshold_config["threshold_label"]

        threshold_df = apply_threshold(base_df, threshold_type, threshold_value)
        
        sample_output_dir = output_dir / sample_id
        sample_output_dir.mkdir(parents=True, exist_ok=True)

        sample_summaries.append({
            "sample_id": sample_id,
            "threshold_type": threshold_type,
            "threshold_value": threshold_value,
            "threshold_label": threshold_label,
            "total_unique_variants": len(df),
            "mutect2_found": int(mutect2_found.sum()),
            "deepvariant_found": int(deepvariant_found.sum()),
            "both_callers_found": int(both_found.sum()),
            "mutect2_only": int((mutect2_found & ~deepvariant_found).sum()),
            "deepvariant_only": int((~mutect2_found & deepvariant_found).sum()),
            "plotted_rows_after_threshold": len(threshold_df),
            "percent_both_callers_retained": len(threshold_df) / int(both_found.sum()) * 100 if int(both_found.sum()) > 0 else None,
        })

        for plot_info in plots:
            if plot_info["plot_name"] not in threshold_config["plot_names"]:
                continue

            x_col = plot_info["x_col"]
            y_col = plot_info["y_col"]
            plot_df = threshold_df[[x_col, y_col]].dropna(subset=[x_col, y_col]).copy()

            if plot_df.empty:
                print(f"Skipping {sample_id} {plot_info['plot_name']} {threshold_label}; no plottable rows")
                continue

            x = plot_df[x_col]
            y = plot_df[y_col]

            axis_min = plot_info["axis_min"]
            axis_max = plot_info["axis_max"]

            if axis_max is None:
                axis_max = max(x.max(), y.max())
                axis_max = axis_max * 1.05 if axis_max > 0 else 1

            plt.figure(figsize=(6, 6))
            plt.scatter(x, y, s=10, alpha=0.5)
            plt.plot([axis_min, axis_max], [axis_min, axis_max], linestyle="--", linewidth=1)

            if threshold_type == plot_info["plot_name"]:
                plt.axvline(threshold_value, linestyle=":", linewidth=1)
                plt.axhline(threshold_value, linestyle=":", linewidth=1)

            plt.xlim(axis_min, axis_max)
            plt.ylim(axis_min, axis_max)
            plt.xlabel(plot_info["x_label"])
            plt.ylabel(plot_info["y_label"])
            plt.title(
                f"{sample_id}: {plot_info['x_label']} vs {plot_info['y_label']}\n"
                f"{threshold_label}; plotted rows = {len(plot_df):,}"
            )

            plt.tight_layout()
            out_png = sample_output_dir / f"{sample_id}_mutect2_vs_deepvariant_{plot_info['plot_name']}_{threshold_label}.png"
            plt.savefig(out_png, dpi=300)

            if show_plots:
                plt.show()
            else:
                plt.close()

            print("Saved:", out_png)

    return sample_summaries


## Run all samples

This saves one folder of plots per sample and a summary CSV for all processed samples.


In [ ]:
summaries = []

for sample_file in sample_files:
    summaries.extend(plot_sample(sample_file))

summary_df = pd.DataFrame(summaries)

summary_csv = output_dir / "10_scatterplot_independent_threshold_summary.csv"
summary_df.to_csv(summary_csv, index=False)

print("Saved summary:", summary_csv)
summary_df.head()
